In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA
import pyarrow.parquet as pq
import pyarrow as pa
import os
import time
import joblib

print("Script started. This will take some time to complete.")
start_time = time.time()

# --- Configuration ---
INPUT_FILEPATH = 'pivoted_data_all_rechunked.parquet'
OUTPUT_FILEPATH = 'pivoted_data_reduced_pca.parquet'
N_COMPONENTS = 200
BATCH_SIZE = 32768

SCALER_PATH = 'full_dataset_scaler.pkl'
PCA_PATH = 'full_dataset_pca.pkl'


# --- Main Logic ---
def main():
    """Main function to run the entire PCA pipeline."""
    if not os.path.exists(INPUT_FILEPATH):
        print(f"Error: The input file was not found at {INPUT_FILEPATH}")
        return

    try:
        parquet_file = pq.ParquetFile(INPUT_FILEPATH)
        
        # --- Check for saved models ---
        if os.path.exists(SCALER_PATH) and os.path.exists(PCA_PATH):
            print("\nFound saved scaler and PCA models. Loading them and skipping Passes 1 and 2.")
            scaler = joblib.load(SCALER_PATH)
            pca = joblib.load(PCA_PATH)
            print("Models loaded successfully.")
        else:
            # --- Pass 1 & 2: Fitting Logic (will be skipped if models exist) ---
            print("Identifying all numeric columns...")
            numeric_cols_to_use = [
                parquet_file.schema.column(i).name 
                for i in range(len(parquet_file.schema))
                if 'INT' in str(parquet_file.schema.column(i).physical_type) or \
                   'FLOAT' in str(parquet_file.schema.column(i).physical_type) or \
                   'DOUBLE' in str(parquet_file.schema.column(i).physical_type)
            ]
            print(f"Found {len(numeric_cols_to_use)} numeric features to process.")
            
            print("\n--- Pass 1 of 3: Fitting the StandardScaler ---")
            scaler = StandardScaler()
            for i in range(parquet_file.num_row_groups):
                print(f"  Processed chunk {i+1}/{parquet_file.num_row_groups} for scaling...")
                df_chunk = parquet_file.read_row_group(i, columns=numeric_cols_to_use).to_pandas()
                df_chunk.fillna(0, inplace=True)
                scaler.partial_fit(df_chunk)
            print("StandardScaler has been fitted.")

            print("\n--- Pass 2 of 3: Fitting the IncrementalPCA model ---")
            pca = IncrementalPCA(n_components=N_COMPONENTS, batch_size=BATCH_SIZE)
            for i in range(parquet_file.num_row_groups):
                print(f"  Processed chunk {i+1}/{parquet_file.num_row_groups} for PCA fitting...")
                df_chunk = parquet_file.read_row_group(i, columns=numeric_cols_to_use).to_pandas()
                df_chunk.fillna(0, inplace=True)
                scaled_chunk = scaler.transform(df_chunk)
                pca.partial_fit(scaled_chunk)
            print("IncrementalPCA model has been fitted.")

            print("\nSaving intermediate scaler and PCA models...")
            joblib.dump(scaler, SCALER_PATH)
            joblib.dump(pca, PCA_PATH)
            print("Intermediate models saved successfully.")

        # --- Pass 3 of 3: Transforming data and saving to new file (SEQUENTIAL) ---
        print("\n--- Pass 3 of 3: Transforming data and saving to new file ---")
        
        output_schema = pa.schema([pa.field(f'PC_{i+1}', pa.float32()) for i in range(N_COMPONENTS)])
        
        numeric_cols_to_use = [
            parquet_file.schema.column(i).name 
            for i in range(len(parquet_file.schema))
            if 'INT' in str(parquet_file.schema.column(i).physical_type) or \
               'FLOAT' in str(parquet_file.schema.column(i).physical_type) or \
               'DOUBLE' in str(parquet_file.schema.column(i).physical_type)
        ]

        with pq.ParquetWriter(OUTPUT_FILEPATH, output_schema) as writer:
            for i in range(parquet_file.num_row_groups):
                print(f"  Transforming and writing chunk {i+1}/{parquet_file.num_row_groups}...")
                
                df_chunk = parquet_file.read_row_group(i, columns=numeric_cols_to_use).to_pandas()
                df_chunk.fillna(0, inplace=True)
                
                scaled_chunk = scaler.transform(df_chunk)
                
                # --- FIX: Cast the output to float32 to match the schema ---
                transformed_chunk = pca.transform(scaled_chunk).astype(np.float32)
                
                table = pa.Table.from_pandas(
                    pd.DataFrame(transformed_chunk, columns=[f'PC_{i+1}' for i in range(N_COMPONENTS)])
                )
                writer.write_table(table)

        print("\nTransformation complete. New file saved successfully.")
        
    except Exception as e:
        print(f"An error occurred during the process: {e}")
        
    end_time = time.time()
    total_minutes = (end_time - start_time) / 60
    print(f"Total script execution time: {total_minutes:.2f} minutes.")


if __name__ == '__main__':
    main()



Script started. This will take some time to complete.

Found saved scaler and PCA models. Loading them and skipping Passes 1 and 2.
Models loaded successfully.

--- Pass 3 of 3: Transforming data and saving to new file ---
  Transforming and writing chunk 1/77...
  Transforming and writing chunk 2/77...
  Transforming and writing chunk 3/77...
  Transforming and writing chunk 4/77...
  Transforming and writing chunk 5/77...
  Transforming and writing chunk 6/77...
  Transforming and writing chunk 7/77...
  Transforming and writing chunk 8/77...
  Transforming and writing chunk 9/77...
  Transforming and writing chunk 10/77...
  Transforming and writing chunk 11/77...
  Transforming and writing chunk 12/77...
  Transforming and writing chunk 13/77...
  Transforming and writing chunk 14/77...
  Transforming and writing chunk 15/77...
  Transforming and writing chunk 16/77...
  Transforming and writing chunk 17/77...
  Transforming and writing chunk 18/77...
  Transforming and writing chu